In [1]:
# ============================================================
# UCI-HAR Classification with ResNet + Transformer
# TensorFlow / Google Colab
# Simplified Version
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. Configuration
# ============================================================

DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR"

SEED = 42
BATCH_SIZE = 128
EPOCHS = 50
VALIDATION_SPLIT = 0.15

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

INPUT_LENGTH = 128
NUM_CHANNELS = 9
NUM_CLASSES = 6

CLASS_NAMES = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING",
]

SIGNAL_NAMES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 3. Load UCI-HAR
# ============================================================

def load_ucihar_split(data_root, split):
    signal_dir = os.path.join(data_root, split, "Inertial Signals")

    X = []
    for name in SIGNAL_NAMES:
        path = os.path.join(signal_dir, f"{name}_{split}.txt")
        X.append(np.loadtxt(path))

    X = np.stack(X, axis=-1).astype(np.float32)

    y_path = os.path.join(data_root, split, f"y_{split}.txt")
    y = np.loadtxt(y_path).astype(np.int64) - 1

    return X, y


X_train, y_train = load_ucihar_split(DATA_ROOT, "train")
X_test, y_test = load_ucihar_split(DATA_ROOT, "test")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

# ============================================================
# 4. Normalization
# ============================================================

mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# ============================================================
# 5. Train / Validation Split
# ============================================================

indices = np.arange(len(X_train))
np.random.shuffle(indices)

val_size = int(len(X_train) * VALIDATION_SPLIT)
val_idx = indices[:val_size]
train_idx = indices[val_size:]

X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
    .shuffle(len(X_tr), seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Train:", X_tr.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)

# ============================================================
# 6. Model
# ============================================================

def residual_block(x, filters, kernel_size=7, dropout=0.1):
    shortcut = x

    x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding="same", use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


class LearnablePositionalEmbedding(layers.Layer):
    def __init__(self, max_len, dim):
        super().__init__()
        self.max_len = max_len
        self.dim = dim

    def build(self, input_shape):
        self.pos_embedding = self.add_weight(
            name="pos_embedding",
            shape=(1, self.max_len, self.dim),
            initializer="random_normal",
            trainable=True,
        )

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_embedding[:, :seq_len, :]


def transformer_block(x, dim=128, heads=4, ff_dim=256, dropout=0.1):
    attn = layers.MultiHeadAttention(
        num_heads=heads,
        key_dim=dim // heads,
        dropout=dropout
    )(x, x)

    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = layers.Dense(ff_dim, activation="relu")(x)
    ffn = layers.Dropout(dropout)(ffn)
    ffn = layers.Dense(dim)(ffn)

    x = layers.LayerNormalization(epsilon=1e-6)(x + ffn)

    return x


def build_model():
    inputs = layers.Input(shape=(INPUT_LENGTH, NUM_CHANNELS))

    x = layers.Conv1D(64, 7, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = residual_block(x, 64)
    x = residual_block(x, 64)
    x = residual_block(x, 128)

    x = layers.Conv1D(128, 1, padding="same")(x)
    x = LearnablePositionalEmbedding(INPUT_LENGTH, 128)(x)

    x = transformer_block(x)
    x = transformer_block(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return models.Model(inputs, outputs)


model = build_model()

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# 7. Train
# ============================================================

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# ============================================================
# 8. Evaluation
# ============================================================

test_loss, test_acc = model.evaluate(test_ds, verbose=0)

y_prob = model.predict(test_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test, y_pred)

macro_precision = precision_score(y_test, y_pred, average="macro")
macro_recall = recall_score(y_test, y_pred, average="macro")
macro_f1 = f1_score(y_test, y_pred, average="macro")

weighted_precision = precision_score(y_test, y_pred, average="weighted")
weighted_recall = recall_score(y_test, y_pred, average="weighted")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("\n==============================")
print("Fixed UCI-HAR Test Result")
print("==============================")
print(f"Test Loss          : {test_loss:.6f}")
print(f"Accuracy           : {acc:.6f}")
print(f"Macro Precision    : {macro_precision:.6f}")
print(f"Macro Recall       : {macro_recall:.6f}")
print(f"Macro F1           : {macro_f1:.6f}")
print(f"Weighted Precision : {weighted_precision:.6f}")
print(f"Weighted Recall    : {weighted_recall:.6f}")
print(f"Weighted F1        : {weighted_f1:.6f}")

print("\n==============================")
print("Classification Report")
print("==============================")
print(classification_report(
    y_test,
    y_pred,
    target_names=CLASS_NAMES,
    digits=4
))

print("\n==============================")
print("Confusion Matrix")
print("==============================")

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=[f"True_{name}" for name in CLASS_NAMES],
    columns=[f"Pred_{name}" for name in CLASS_NAMES]
)

print(cm_df)
display(cm_df)

TensorFlow: 2.20.0
Mounted at /content/drive
X_train: (7352, 128, 9)
X_test : (2947, 128, 9)
Train: (6250, 128, 9)
Val  : (1102, 128, 9)
Test : (2947, 128, 9)
Epoch 1/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 70s 697ms/step - accuracy: 0.8818 - loss: 0.3169 - val_accuracy: 0.8439 - val_loss: 0.4716 - learning_rate: 0.0010
Epoch 2/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9406 - loss: 0.1508 - val_accuracy: 0.9492 - val_loss: 0.1402 - learning_rate: 0.0010
Epoch 3/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9474 - loss: 0.1300 - val_accuracy: 0.9465 - val_loss: 0.1492 - learning_rate: 0.0010
Epoch 4/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9418 - loss: 0.1512 - val_accuracy: 0.9510 - val_loss: 0.1182 - learning_rate: 0.0010
Epoch 5/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9422 - loss: 0.1384 - val_accuracy: 0.9083 - val_loss: 0.3458 - learning_rate: 0.0010
Epoch 6/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.9526 - loss: 0.1152

,Pred_WALKING,Pred_WALKING_UPSTAIRS,Pred_WALKING_DOWNSTAIRS,Pred_SITTING,Pred_STANDING,Pred_LAYING
True_WALKING,493,0,3,0,0,0
True_WALKING_UPSTAIRS,0,446,25,0,0,0
True_WALKING_DOWNSTAIRS,0,0,420,0,0,0
True_SITTING,0,3,0,411,77,0
True_STANDING,0,0,0,69,463,0
True_LAYING,0,0,0,0,0,537
